In [4]:
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.express as px
import pandas as pd
import numpy as np

# --- Initialisation de l'app Dash ---
app = dash.Dash(__name__)
server = app.server

# --- Données initiales ---
# Coordonnées approximatives de quelques villes pour simuler les offres
villes = pd.DataFrame({
    "ville": ["Paris", "Lyon", "Marseille", "Lille", "Bordeaux"],
    "lat": [48.8566, 45.764, 43.2965, 50.6292, 44.8378],
    "lon": [2.3522, 4.8357, 5.3698, 3.0573, -0.5792],
})

# DataFrame vide pour les offres simulées
offres = pd.DataFrame(columns=["ville", "lat", "lon", "type"])

# --- Layout ---
app.layout = html.Div([
    html.H1("Simulation Marché Immobilier d'Entreprise"),
    html.Div([
        dcc.Graph(id='carte'),
    ], style={'width': '70%', 'display': 'inline-block', 'vertical-align': 'top'}),
    html.Div([
        html.H4("Indicateurs"),
        html.Div(id='indicateurs'),
        html.H4("Console"),
        html.Div(id='console', style={'height': '400px', 'overflowY': 'scroll', 
                                      'border': '1px solid black', 'padding': '5px'})
    ], style={'width': '28%', 'display': 'inline-block', 'padding-left': '2%'}),
    
    # Interval pour mise à jour chaque seconde
    dcc.Interval(id='interval-component', interval=1000, n_intervals=0)
])

# --- Callbacks ---
@app.callback(
    Output('carte', 'figure'),
    Output('indicateurs', 'children'),
    Output('console', 'children'),
    Input('interval-component', 'n_intervals')
)
def update_simulation(n):
    global offres
    
    # --- Simuler de nouvelles offres ---
    # Chaque semaine (seconde), il y a 0-2 nouvelles offres par ville
    nouvelles_offres = []
    for idx, ville in villes.iterrows():
        nb_offres = np.random.poisson(0.5)  # moyenne 0.5 nouvelles offres
        for _ in range(nb_offres):
            nouvelles_offres.append({
                "ville": ville["ville"],
                "lat": ville["lat"] + np.random.uniform(-0.01, 0.01),
                "lon": ville["lon"] + np.random.uniform(-0.01, 0.01),
                "type": "offre"
            })
    if nouvelles_offres:
        offres = pd.concat([offres, pd.DataFrame(nouvelles_offres)], ignore_index=True)

    # --- Simuler des transactions ---
    nb_transactions = np.random.poisson(0.3)
    transactions = []
    if not offres.empty:
        for _ in range(nb_transactions):
            idx_choice = np.random.choice(offres.index)
            transac = offres.loc[idx_choice].copy()
            transac["type"] = "transaction"
            transactions.append(transac)
            # supprimer de la liste des offres
            offres = offres.drop(idx_choice)
    
    # --- Mettre à jour la figure ---
    df_map = pd.concat([offres, pd.DataFrame(transactions)] if transactions else [offres])
    fig = px.scatter_mapbox(
        df_map,
        lat="lat",
        lon="lon",
        color="type",
        hover_name="ville",
        zoom=4,
        height=600
    )
    fig.update_layout(mapbox_style="open-street-map")
    
    # --- Indicateurs ---
    indicateurs = [
        html.P(f"Nombre d'offres : {len(offres)}"),
        html.P(f"Nombre de transactions cette semaine : {len(transactions)}")
    ]
    
    # --- Console ---
    console_messages = []
    for off in nouvelles_offres:
        console_messages.append(f"Semaine {n+1} : Nouvelle offre à {off['ville']}")
    for tr in transactions:
        console_messages.append(f"Semaine {n+1} : Transaction réalisée à {tr['ville']}")
    
    # On conserve les messages précédents
    if hasattr(update_simulation, "console_history"):
        update_simulation.console_history += console_messages
    else:
        update_simulation.console_history = console_messages
    
    # Limiter le nombre de messages visibles
    console_content = [html.P(msg) for msg in update_simulation.console_history[-20:]]
    
    return fig, indicateurs, console_content

# --- Lancer l'app ---
if __name__ == '__main__':
    app.run(debug=True)


C:\Users\quent\AppData\Local\Temp\ipykernel_10876\1457847760.py:64: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

C:\Users\quent\AppData\Local\Temp\ipykernel_10876\1457847760.py:80: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/

C:\Users\quent\AppData\Local\Temp\ipykernel_10876\1457847760.py:80: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/

C:\Users\quent\AppData\Local\Temp\ipykernel_10876\1457847760.py:80: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/

C:\Use